In [3]:
from pathlib import Path

import pandas as pd
import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt


def main():
    # Rutas relativas a la ubicación de este script.
    base = Path.cwd().parent
    data = base / "data"
    submission = base / "submission"
    submission.mkdir(parents=True, exist_ok=True)

    # Leer los archivos.
    drivers = pd.read_csv(data / "drivers.csv")
    timesheet = pd.read_csv(data / "timesheet.csv")

    # Calcular los totales de cada conductor.
    totals = (
        timesheet.groupby("driverId", as_index=False)
        .agg({
            "hours-logged": "sum",
            "miles-logged": "sum",
        })
    )

    # Agregar el nombre sin incluir datos sensibles como el SSN.
    summary = totals.merge(
        drivers[["driverId", "name"]],
        on="driverId",
        how="left",
        validate="many_to_one",
    )

    summary = summary[
        ["driverId", "name", "hours-logged", "miles-logged"]
    ].sort_values(
        ["miles-logged", "driverId"],
        ascending=[False, True],
    )

    # Guardar el resumen.
    summary.to_csv(submission / "summary.csv", index=False)

    # Graficar los 10 conductores con más millas.
    top10 = summary.head(10).sort_values("miles-logged")
    labels = (
        top10["name"].fillna("Sin nombre")
        + " ("
        + top10["driverId"].astype(str)
        + ")"
    )

    fig, ax = plt.subplots(figsize=(10, 6))
    ax.barh(labels, top10["miles-logged"], color="#2685BF")
    ax.set_title("Top 10 conductores por millas recorridas")
    ax.set_xlabel("Millas totales")
    ax.set_ylabel("Conductor")
    fig.tight_layout()
    fig.savefig(submission / "top10_drivers.png", dpi=200)
    plt.close(fig)

    print(f"Archivos generados en: {submission}")


if __name__ == "__main__":
    main()

Archivos generados en: c:\Users\jg702\Documents\GitHub\std-jennifer-gomez-sanchez\PRE_08_drivers_sqlite\submission
